# 안전귀가Navi — Day 4 점수 엔진

- 입력: `search.analyze_lat_lon(lat, lon)` 신호 dict
- 출력: 0~100 종합 점수 + 4개 활성 카테고리별 점수 + 등급 + 한 줄 요약 + 추천 행동
- 6개 항목 중 4개 활성(cctv/light/bell/policy), 2개 placeholder(귀가/시간환경)

**API 키 없이 좌표 직접 입력으로 검증.**

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path(r'C:\Users\pjhic\Projects\Seoul_bigdata\src')))
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import search
import scoring

print('활성 카테고리:', [scoring.CATEGORIES[k] for k in scoring.ACTIVE_CATEGORIES])
print('비활성 카테고리:', [scoring.CATEGORIES[k] for k in scoring.INACTIVE_CATEGORIES])
print('priority 옵션:', list(scoring.PRIORITY_WEIGHTS.keys()))

활성 카테고리: ['감시 인프라', '야간 조명', '긴급 대응', '정책 접근성']
비활성 카테고리: ['귀가 접근성', '시간·환경 보정']
priority 옵션: ['balanced', 'cctv', 'lighting', 'emergency', 'transport']


## 1. 샘플 좌표 신호 수집
지하철역·자치구 다양한 위치를 커버.

In [2]:
samples = {
    '강남역':         (37.4979, 127.0276),
    '신촌역':         (37.5559, 126.9367),
    '잠실역':         (37.5133, 127.1000),    # 송파구 — 데이터 적은 곳
    '관악구 신림동':   (37.4843, 126.9296),
    '종로구 자하문로': (37.5814, 126.9686),
    '강동구 천호동':   (37.5384, 127.1239),    # 강동구 — CCTV 보강 확인
    '성동구 행당동':   (37.5573, 127.0395),    # 성동구 — CCTV 보강 확인
}

signals = {}
for name, (lat, lon) in samples.items():
    signals[name] = search.analyze_lat_lon(lat, lon)

print(f'{len(signals)}개 좌표 신호 수집 완료')

7개 좌표 신호 수집 완료


## 2. balanced priority로 점수 산출

In [3]:
import pandas as pd

rows = []
for name, sig in signals.items():
    r = scoring.score_signal(sig, priority='balanced')
    cs = r['category_scores']
    rows.append({
        '주소': name,
        '종합': r['total_score'],
        '등급': r['grade'],
        '감시': cs['surveillance']['score'],
        '조명': cs['lighting']['score'],
        '긴급': cs['emergency']['score'],
        '정책': cs['policy']['score'],
    })

df = pd.DataFrame(rows).sort_values('종합', ascending=False)
df

,주소,종합,등급,감시,조명,긴급,정책
1,신촌역,88,매우 양호,100,70,94,86
3,관악구 신림동,78,양호,100,42,94,68
4,종로구 자하문로,77,양호,100,51,75,77
5,강동구 천호동,74,양호,77,42,100,77
6,성동구 행당동,65,양호,94,33,94,26
0,강남역,64,보통,85,33,94,35
2,잠실역,40,보완 필요,70,33,24,26


## 3. 단일 주소 상세 — 한 줄 요약·추천 행동

In [4]:
for name in ['강남역', '잠실역', '강동구 천호동']:
    r = scoring.score_signal(signals[name], priority='balanced')
    print(f'\n=== {name} ===')
    print(f'종합 안심 점수: {r["total_score"]} / 100  ({r["grade"]})')
    print(f'한 줄 요약: {r["one_line_summary"]}')
    print(f'추천 행동: {r["recommended_action"]}')
    print(f'\n  카테고리별 점수:')
    for key, cat in r['category_scores'].items():
        if cat.get('active'):
            ev = cat['evidence']
            ev_str = ', '.join(f'{k}={v}' for k, v in ev.items() if v is not None)
            print(f'    {cat["name"]:<12} {cat["score"]:>3}점 (가중치 {cat["weight"]:.2f})  | {ev_str}')
        else:
            print(f'    {cat["name"]:<12}  -  ({cat["note"]})')


=== 강남역 ===
종합 안심 점수: 64 / 100  (보통)
한 줄 요약: 긴급 대응은 양호하지만 야간 조명은 보완이 필요한 주소입니다 (종합 64점).
추천 행동: 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.

  카테고리별 점수:
    감시 인프라        85점 (가중치 0.30)  | nearest_m=122.21858349519921, count_300m=49, count_500m=121
    야간 조명         33점 (가중치 0.25)  | nearest_m=583.9041908569873, count_100m=0, count_300m=0
    긴급 대응         94점 (가중치 0.25)  | nearest_m=99.0963976756277, count_300m=8, count_500m=27
    정책 접근성        35점 (가중치 0.20)  | nearest_route_m=558.5588729005665, nearest_route_name=강남안심01, routes_within_300m=0, routes_within_500m=0, facility_count_300m=0
    귀가 접근성        -  (MVP 미적용 — 정류장 데이터 추가 시 활성화 예정)
    시간·환경 보정      -  (MVP 미적용 — 생활인구·기상·일몰 데이터 추가 시 활성화 예정)

=== 잠실역 ===
종합 안심 점수: 40 / 100  (보완 필요)
한 줄 요약: 감시 인프라는 양호하지만 긴급 대응은 보완이 필요한 주소입니다 (종합 40점).
추천 행동: 긴급 대응 인프라 접근성이 보완될 여지가 있어, 가장 가까운 안전비상벨·안심지킴이집 위치를 미리 확인해두는 것이 좋습니다.

  카테고리별 점수:
    감시 인프라        70점 (가중치 0.30)  | nearest_m=55.51684732118178, count_300m=3, c

## 4. priority별 점수 변화
사용자 우선순위에 따라 가중치가 달라지면 같은 주소의 종합 점수도 달라져야 함.

In [5]:
rows = []
for priority in ['balanced', 'cctv', 'lighting', 'emergency']:
    row = {'priority': priority}
    for name in ['강남역', '잠실역', '관악구 신림동']:
        r = scoring.score_signal(signals[name], priority=priority)
        row[name] = r['total_score']
    rows.append(row)

pd.DataFrame(rows).set_index('priority')

,강남역,잠실역,관악구 신림동
priority,,,
balanced,64,40,78
cctv,69,47,82
lighting,56,38,68
emergency,71,35,81


## 5. 강동구·성동구 보강 효과 검증
안심귀갓길 시설물(시설코드 302)로 보강한 자치구가 0점이 안 나오는지 확인.

In [6]:
for name in ['강동구 천호동', '성동구 행당동']:
    r = scoring.score_signal(signals[name])
    cctv = r['category_scores']['surveillance']
    print(f'{name}: 감시 인프라 {cctv["score"]}점  (최근접 CCTV {cctv["evidence"]["nearest_m"]}m, 300m 내 {cctv["evidence"]["count_300m"]}개)')
print('\n→ 둘 다 0점 회피 확인.')

강동구 천호동: 감시 인프라 77점  (최근접 CCTV 121.35338476956568m, 300m 내 10개)
성동구 행당동: 감시 인프라 94점  (최근접 CCTV 65.54548084354822m, 300m 내 33개)

→ 둘 다 0점 회피 확인.


## 6. 송파구(잠실역) 거리 기반 점수 회복 확인
안전비상벨이 1건뿐이어도 거리 기반 신호로 점수 산출되는지 확인.

In [7]:
r = scoring.score_signal(signals['잠실역'])
bell = r['category_scores']['emergency']
print(f'잠실역 긴급 대응 점수: {bell["score"]}점')
print(f'  거리 기반: {bell["components"]["distance_score"]}, 밀도 기반: {bell["components"]["density_score"]}')
print(f'  최근접 비상벨: {bell["evidence"]["nearest_m"]}m, 300m 내 {bell["evidence"]["count_300m"]}개')
print('\n→ 데이터 적은 자치구도 거리 신호로 점수 산출 ✓')

잠실역 긴급 대응 점수: 24점
  거리 기반: 30, 밀도 기반: 15
  최근접 비상벨: 904.4818531592655m, 300m 내 0개

→ 데이터 적은 자치구도 거리 신호로 점수 산출 ✓


## 7. 전체 출력 형식 — Day 5 AI 리포트 입력으로 사용될 구조

In [8]:
result = scoring.score_signal(signals['관악구 신림동'], priority='balanced')
result['address'] = '서울특별시 관악구 신림동 (샘플)'
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "lat": 37.4843,
  "lon": 126.9296,
  "address": "서울특별시 관악구 신림동 (샘플)",
  "priority": "balanced",
  "total_score": 78,
  "grade": "양호",
  "one_line_summary": "감시 인프라는 양호하지만 야간 조명은 보완이 필요한 주소입니다 (종합 78점).",
  "recommended_action": "집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.",
  "category_scores": {
    "surveillance": {
      "score": 100,
      "components": {
        "distance_score": 100,
        "density_score": 100
      },
      "evidence": {
        "nearest_m": 30.18687597926184,
        "count_300m": 43,
        "count_500m": 117
      },
      "active": true,
      "weight": 0.3,
      "name": "감시 인프라"
    },
    "lighting": {
      "score": 42,
      "components": {
        "distance_score": 60,
        "density_score": 15
      },
      "evidence": {
        "nearest_m": 252.26837250622887,
        "count_100m": 0,
        "count_300m": 3
      },
      "active": true,
      "weight": 0.25,
      "name": "야간 조명"
    },
    "emergency": {
      "s